# Notebook 08 — Decision & Recommendation Engine

## Objective
Generate explainable investigation suggestions from the consolidated
ward profiles using explicit, configurable rules.

## Workflow
1. Load and validate the ward profiles.
2. Define rules, thresholds, and eligibility conditions.
3. Generate suggestions with supporting counts and rates.
4. Inspect results and test threshold sensitivity.
5. Export suggestions and rule metadata.

## Interpretation and limitations
- Suggestions require human review.
- Thresholds are exploratory and configurable.
- Anomaly rankings and clusters provide context, not operational priority.
- Preserve records with missing model outputs and explain exclusions.
- Geographic mappings and supply timing compatibility remain unverified.
- Do not infer driver shortages, causes, or expected business improvements.

## 8.1 — Setup and Data Loading

Load the ward profiles exported by Notebook 07.
Inspect their structure before defining recommendation rules.

In [2]:
from pathlib import Path
import pandas as pd

PROFILE_PATH = Path(
    "/content/NammaFlow/reports/ward_level_intelligence/ward_profiles.csv"
)

print("Profile file found:", PROFILE_PATH.exists())

if PROFILE_PATH.exists():
    ward_profiles = pd.read_csv(PROFILE_PATH)

    print("Shape:", ward_profiles.shape)
    print("\nColumns:")
    print(ward_profiles.columns.tolist())

    display(ward_profiles.head())
else:
    print(
        "The project files are not available at this path. "
        "We need to locate or load them before continuing."
    )

Profile file found: False
The project files are not available at this path. We need to locate or load them before continuing.


In [3]:
from pathlib import Path
import subprocess

PROJECT_ROOT = Path("/content/NammaFlow")

if not PROJECT_ROOT.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/Narasimhan15/NammaFlow.git",
            str(PROJECT_ROOT),
        ],
        check=True,
    )
else:
    print("Project folder already exists.")

PROFILE_PATH = (
    PROJECT_ROOT / "reports/ward_level_intelligence/ward_profiles.csv"
)

print("Profile file found:", PROFILE_PATH.exists())

if PROFILE_PATH.exists():
    ward_profiles = pd.read_csv(PROFILE_PATH)
    print("Shape:", ward_profiles.shape)
    print("\nColumns:", ward_profiles.columns.tolist())
    display(ward_profiles.head())
else:
    print("ward_profiles.csv is missing from the expected repository path.")

Profile file found: True
Shape: (244, 74)

Columns: ['ward_num', 'ward_searches', 'ward_searches_got_e', 'ward_q_requests', 'ward_q_received', 'ward_search_to_e_rate', 'ward_e_to_q_request_rate', 'ward_q_accept_rate', 'ward_search_to_q_request_rate', 'ward_search_to_q_received_rate', 'ward_q_supply_gap', 'ward_q_supply_gap_rate', 'ward_search_to_q_received_gap', 'ward_bookings', 'ward_rides', 'ward_completed_rides', 'ward_cancellations', 'ward_driver_cancellations', 'ward_rider_cancellations', 'ward_earnings', 'ward_booking_to_ride_rate', 'ward_ride_completion_rate', 'ward_booking_completion_rate', 'ward_cancel_per_booking', 'ward_other_cancellations', 'ward_driver_cancel_share', 'ward_rider_cancel_share', 'ward_other_cancel_share', 'ward_earning_per_ride', 'ward_earning_per_completed_ride', 'ward_booking_to_ride_gap', 'ward_ride_completion_gap', 'ward_booking_to_completion_gap', 'ward_ride_booking_mismatch', 'supply_vehicle_coverage', 'active_drvr_auto_cab', 'notonride_drvr_auto_cab',

,ward_num,ward_searches,ward_searches_got_e,ward_q_requests,ward_q_received,ward_search_to_e_rate,ward_e_to_q_request_rate,ward_q_accept_rate,ward_search_to_q_request_rate,ward_search_to_q_received_rate,...,pressure_score_sum,exploratory_cluster,anomaly_score,review_position,exclusion_reason,has_model_results,low_booking_count,is_w_0,geographic_mapping_status,share_of_all_searches
0,b_1,72,72,39,21,1.0,0.541667,0.538462,0.541667,0.291667,...,0.461538,1.0,0.413388,165.0,NaN,True,False,False,unverified,0.001994
1,b_10,188,188,139,65,1.0,0.739362,0.467626,0.739362,0.345745,...,0.532374,0.0,0.399612,200.0,NaN,True,False,False,unverified,0.005206
2,b_100,159,159,129,56,1.0,0.811321,0.434109,0.811321,0.352201,...,0.565891,0.0,0.431429,130.0,NaN,True,False,False,unverified,0.004403
3,b_101,43,43,29,19,1.0,0.674419,0.655172,0.674419,0.441860,...,0.344828,2.0,0.482583,58.0,NaN,True,False,False,unverified,0.001191
4,b_102,127,127,110,52,1.0,0.866142,0.472727,0.866142,0.409449,...,0.527273,0.0,0.403445,194.0,NaN,True,False,False,unverified,0.003517


## 8.2 — Input Validation

Check unique identifiers, nonnegative counts, and agreement between
stored rates and their underlying counts. Preserve all records,
including those excluded from modelling.

In [4]:
import numpy as np

df = ward_profiles.copy()

assert df["ward_num"].notna().all(), "Missing ward identifiers"
assert df["ward_num"].is_unique, "Duplicate ward identifiers"

count_cols = [
    "ward_searches",
    "ward_bookings",
    "ward_completed_rides",
]

for col in count_cols:
    assert pd.api.types.is_numeric_dtype(df[col]), f"Non-numeric: {col}"
    assert np.isfinite(df[col]).all(), f"Missing/infinite counts: {col}"
    assert df[col].ge(0).all(), f"Negative counts: {col}"
    assert df[col].eq(df[col].round()).all(), f"Non-integer counts: {col}"

rate_checks = [
    ("ward_search_to_booking_rate", "ward_bookings", "ward_searches"),
    ("ward_booking_completion_rate", "ward_completed_rides", "ward_bookings"),
]

for rate, numerator, denominator in rate_checks:
    expected = df[numerator].div(df[denominator].replace(0, np.nan))
    assert np.allclose(
        df[rate], expected, equal_nan=True, rtol=1e-6, atol=1e-9
    ), f"Rate/count mismatch: {rate}"

print("Identifier, count, and rate consistency checks passed.")
print("Total records:", len(df))
print("Zero-booking records:", df["ward_bookings"].eq(0).sum())
print("w_0 records:", df["ward_num"].eq("w_0").sum())

print("\nModel coverage:")
print(df["has_model_results"].value_counts(dropna=False))

print("\nGeographic mapping:")
print(df["geographic_mapping_status"].value_counts(dropna=False))

display(
    df.loc[
        df["ward_bookings"].eq(0),
        ["ward_num", "ward_bookings",
         "ward_booking_completion_rate", "exclusion_reason"]
    ]
)

Identifier, count, and rate consistency checks passed.
Total records: 244
Zero-booking records: 3
w_0 records: 1

Model coverage:
has_model_results
True     241
False      3
Name: count, dtype: int64

Geographic mapping:
geographic_mapping_status
unverified    244
Name: count, dtype: int64


,ward_num,ward_bookings,ward_booking_completion_rate,exclusion_reason
76,b_168,0,NaN,nonpositive_bookings; missing_or_nonfinite_input
162,b_26,0,NaN,nonpositive_bookings; missing_or_nonfinite_input
182,b_44,0,NaN,nonpositive_bookings; missing_or_nonfinite_input


## 8.3 — Investigation Rules and Initial Thresholds

These configurable thresholds are exploratory choices, not validated
operational standards. Multiple rules may apply to one record.

### R01 — Search-to-booking investigation
Conditions:
- At least 100 searches.
- Search-to-booking rate below 15%.
- Exclude w_0 from this comparison.

Suggestion:
Inspect the search-to-booking funnel and available event records.
The observed rate does not establish the cause of lost conversions.

### R02 — Booking-to-completion investigation
Conditions:
- At least 10 bookings.
- Booking completion rate below 50%.
- Exclude w_0 from this comparison.

Suggestion:
Inspect ride progression and cancellation records.
The observed rate does not establish responsibility or cause.

### R03 — Limited booking evidence
Condition:
- Fewer than 10 bookings.

Suggestion:
Review underlying counts before interpreting completion rates.
For zero bookings, completion rate is undefined.

### R04 — Unresolved identifier
Condition:
- Identifier equals w_0.

Suggestion:
Verify the identifier's meaning before geographic interpretation.
Retain this record separately for context.

### Shared interpretation
- Preserve all 244 records.
- Missing model results do not prevent count-based rules.
- Clusters and anomaly scores provide supplementary context only.
- No rule establishes driver shortages or operational priority.
- Geographic mapping remains unverified for every record.
- Test threshold sensitivity before finalizing outputs.

In [5]:
RULE_CONFIG = {
    "R01": {
        "min_searches": 100,
        "conversion_below": 0.15,
    },
    "R02": {
        "min_bookings": 10,
        "completion_below": 0.50,
    },
    "R03": {
        "bookings_below": 10,
    },
    "R04": {
        "identifier": "w_0",
    },
}

display(
    pd.DataFrame([
        {"rule_id": rule_id, **settings}
        for rule_id, settings in RULE_CONFIG.items()
    ]).set_index("rule_id")
)

,min_searches,conversion_below,min_bookings,completion_below,bookings_below,identifier
rule_id,,,,,,
R01,100.0,0.15,NaN,NaN,NaN,NaN
R02,NaN,NaN,10.0,0.5,NaN,NaN
R03,NaN,NaN,NaN,NaN,10.0,NaN
R04,NaN,NaN,NaN,NaN,NaN,w_0


## 8.4 — Apply Investigation Rules

Create one row per triggered rule, including evidence and a suggested
investigation. Retain the complete profile table separately.
A record may trigger multiple rules; trigger counts are not priorities.

In [6]:
def apply_rules(profiles, config):
    special_id = config["R04"]["identifier"]
    comparable = profiles["ward_num"].ne(special_id)

    masks = {
        "R01": (
            comparable
            & profiles["ward_searches"].ge(config["R01"]["min_searches"])
            & profiles["ward_search_to_booking_rate"].lt(
                config["R01"]["conversion_below"]
            )
        ),
        "R02": (
            comparable
            & profiles["ward_bookings"].ge(config["R02"]["min_bookings"])
            & profiles["ward_booking_completion_rate"].lt(
                config["R02"]["completion_below"]
            )
        ),
        "R03": profiles["ward_bookings"].lt(
            config["R03"]["bookings_below"]
        ),
        "R04": profiles["ward_num"].eq(special_id),
    }

    suggestions = {
        "R01": "Inspect search-to-booking funnel and event records.",
        "R02": "Inspect ride progression and cancellation records.",
        "R03": (
            "Review counts before interpreting completion rates; "
            "completion rate is undefined for zero bookings."
        ),
        "R04": "Verify identifier meaning before geographic interpretation.",
    }

    evidence_cols = [
        "ward_num", "ward_searches", "ward_bookings",
        "ward_completed_rides", "ward_search_to_booking_rate",
        "ward_booking_completion_rate", "has_model_results",
        "exclusion_reason", "geographic_mapping_status",
        "exploratory_cluster", "anomaly_score",
    ]

    parts = []
    for rule_id, mask in masks.items():
        part = profiles.loc[mask, evidence_cols].copy()
        part.insert(1, "rule_id", rule_id)
        part["suggested_investigation"] = suggestions[rule_id]
        part["limitations"] = (
            "Exploratory rule; human review required. "
            "Geography unverified; causes, shortages and uplift not established. "
            "Model outputs are context only."
        )
        parts.append(part)

    return pd.concat(parts, ignore_index=True)


investigations = apply_rules(df, RULE_CONFIG)

# Keep every input record, including those with no triggered rule.
record_review = df.copy()
trigger_counts = investigations.groupby("ward_num").size()
record_review["triggered_rule_count"] = (
    record_review["ward_num"].map(trigger_counts).fillna(0).astype(int)
)

print("Records retained:", len(record_review))
print("Records with at least one trigger:",
      record_review["triggered_rule_count"].gt(0).sum())
print("Total rule triggers:", len(investigations))

print("\nTriggers by rule:")
display(
    investigations.groupby("rule_id").size()
    .reindex(RULE_CONFIG, fill_value=0)
    .rename("trigger_count").to_frame()
)

display(
    investigations[
        ["ward_num", "rule_id", "ward_searches", "ward_bookings",
         "ward_completed_rides", "ward_search_to_booking_rate",
         "ward_booking_completion_rate"]
    ].head(15)
)

Records retained: 244
Records with at least one trigger: 147
Total rule triggers: 149

Triggers by rule:


,trigger_count
rule_id,
R01,2
R02,76
R03,70
R04,1


,ward_num,rule_id,ward_searches,ward_bookings,ward_completed_rides,ward_search_to_booking_rate,ward_booking_completion_rate
0,b_131,R01,516,69,29,0.133721,0.420290
1,b_181,R01,4862,556,216,0.114356,0.388489
2,b_10,R02,188,65,32,0.345745,0.492308
3,b_101,R02,43,19,5,0.441860,0.263158
4,b_102,R02,127,50,23,0.393701,0.460000
5,b_104,R02,70,21,7,0.300000,0.333333
6,b_107,R02,70,24,11,0.342857,0.458333
7,b_108,R02,166,64,29,0.385542,0.453125
8,b_111,R02,67,20,9,0.298507,0.450000
9,b_112,R02,100,46,22,0.460000,0.478261


## 8.5 — Threshold Sensitivity

Vary one threshold at a time while keeping other settings fixed.
Compare triggered identifier sets with the initial configuration.

This measures dependence on threshold choices, not rule accuracy.
Thresholds are not selected to achieve a desired number of suggestions.

In [7]:
from copy import deepcopy

variations = [
    ("R01", "min_searches", [50, 100, 200]),
    ("R01", "conversion_below", [0.10, 0.15, 0.20]),
    ("R02", "min_bookings", [10, 20, 30]),
    ("R02", "completion_below", [0.40, 0.50, 0.60]),
    ("R03", "bookings_below", [5, 10, 20]),
]

baseline_sets = {
    rule: set(
        investigations.loc[
            investigations["rule_id"].eq(rule), "ward_num"
        ]
    )
    for rule in RULE_CONFIG
}

rows = []

for rule, parameter, values in variations:
    baseline_ids = baseline_sets[rule]

    for value in values:
        config = deepcopy(RULE_CONFIG)
        config[rule][parameter] = value
        result = apply_rules(df, config)

        ids = set(
            result.loc[result["rule_id"].eq(rule), "ward_num"]
        )
        union = ids | baseline_ids

        rows.append({
            "rule_id": rule,
            "parameter": parameter,
            "value": value,
            "is_baseline": value == RULE_CONFIG[rule][parameter],
            "trigger_count": len(ids),
            "added_vs_baseline": len(ids - baseline_ids),
            "removed_vs_baseline": len(baseline_ids - ids),
            "jaccard_vs_baseline": (
                len(ids & baseline_ids) / len(union)
                if union else 1.0
            ),
        })

sensitivity = pd.DataFrame(rows)
display(sensitivity)

,rule_id,parameter,value,is_baseline,trigger_count,added_vs_baseline,removed_vs_baseline,jaccard_vs_baseline
0,R01,min_searches,50.00,False,3,1,0,0.666667
1,R01,min_searches,100.00,True,2,0,0,1.000000
2,R01,min_searches,200.00,False,2,0,0,1.000000
3,R01,conversion_below,0.10,False,0,0,2,0.000000
4,R01,conversion_below,0.15,True,2,0,0,1.000000
5,R01,conversion_below,0.20,False,4,2,0,0.500000
6,R02,min_bookings,10.00,True,76,0,0,1.000000
7,R02,min_bookings,20.00,False,57,0,19,0.750000
8,R02,min_bookings,30.00,False,37,0,39,0.486842
9,R02,completion_below,0.40,False,30,0,46,0.394737


**Sensitivity findings**

Trigger membership depends materially on the selected thresholds,
particularly the booking-completion cutoff and minimum booking count

**Threshold Sensitivity — Findings**

- R01 flags 0, 2, or 4 records at conversion cutoffs of 10%, 15%,
  or 20%, respectively, with the minimum fixed at 100 searches.
- R02 flags 30, 76, or 131 records at completion cutoffs of 40%,
  50%, or 60%, respectively, with the minimum fixed at 10 bookings.
- Raising R02's minimum bookings from 10 to 30 reduces flags from
  76 to 37 at the 50% completion cutoff.
- R03 flags 35, 70, or 130 records using booking cutoffs of 5, 10,
  or 20, respectively.

Results depend materially on threshold choices. Retain the initial
configuration as an exploratory baseline, not an operational standard.
These checks measure sensitivity, not accuracy or business impact.

## 8.6 — Output Integrity Checks

Verify record preservation, unique record–rule pairs, rule eligibility,
and retention of records excluded from modelling.

In [8]:
assert len(record_review) == len(df)
assert set(record_review["ward_num"]) == set(df["ward_num"])
assert not investigations.duplicated(["ward_num", "rule_id"]).any()
assert set(investigations["ward_num"]).issubset(set(df["ward_num"]))
assert record_review["triggered_rule_count"].sum() == len(investigations)

# Independently verify each rule's expected identifier set.
special_id = RULE_CONFIG["R04"]["identifier"]
comparable = df["ward_num"].ne(special_id)

expected = {
    "R01": set(df.loc[
        comparable
        & df["ward_searches"].ge(RULE_CONFIG["R01"]["min_searches"])
        & df["ward_search_to_booking_rate"].lt(
            RULE_CONFIG["R01"]["conversion_below"]
        ),
        "ward_num"
    ]),
    "R02": set(df.loc[
        comparable
        & df["ward_bookings"].ge(RULE_CONFIG["R02"]["min_bookings"])
        & df["ward_booking_completion_rate"].lt(
            RULE_CONFIG["R02"]["completion_below"]
        ),
        "ward_num"
    ]),
    "R03": set(df.loc[
        df["ward_bookings"].lt(RULE_CONFIG["R03"]["bookings_below"]),
        "ward_num"
    ]),
    "R04": set(df.loc[df["ward_num"].eq(special_id), "ward_num"]),
}

assert set(investigations["rule_id"]).issubset(RULE_CONFIG)

for rule_id, expected_ids in expected.items():
    actual_ids = set(investigations.loc[
        investigations["rule_id"].eq(rule_id), "ward_num"
    ])
    assert actual_ids == expected_ids, f"Mismatch: {rule_id}"

zero_booking_ids = set(
    df.loc[df["ward_bookings"].eq(0), "ward_num"]
)
assert zero_booking_ids.issubset(expected["R03"])

print("All output integrity checks passed.")

print("\nRecords triggering multiple rules:")
display(
    record_review.loc[
        record_review["triggered_rule_count"].gt(1),
        ["ward_num", "triggered_rule_count"]
    ]
)

print("\nZero-booking records retained:")
display(
    record_review.loc[
        record_review["ward_num"].isin(zero_booking_ids),
        ["ward_num", "ward_bookings",
         "has_model_results", "triggered_rule_count"]
    ]
)

All output integrity checks passed.

Records triggering multiple rules:


,ward_num,triggered_rule_count
36,b_131,2
91,b_181,2



Zero-booking records retained:


,ward_num,ward_bookings,has_model_results,triggered_rule_count
76,b_168,0,False,1
162,b_26,0,False,1
182,b_44,0,False,1


**Output Integrity — Findings**

All checks passed. All 244 records were preserved, with no duplicate
record–rule pairs. Rule outputs matched their eligibility conditions.

b_131 and b_181 each triggered R01 and R02. The three zero-booking
records remained included and triggered R03, while their completion
rates remained undefined.

## 8.7 — Export Reports and Metadata

Save investigation suggestions, all retained profiles, threshold
sensitivity results, and rule definitions. Record the input hash
and repository revision for traceability.

CSV rates remain fractions. Missing values remain blank.
Trigger counts do not represent severity or operational priority.

In [9]:
import hashlib
import json
import platform
from datetime import datetime, timezone

EXPORT_DIR = PROJECT_ROOT / "reports/decision_recommendation"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

rule_definitions = {
    "R01": {
        "conditions": "searches >= min_searches AND conversion < conversion_below",
        "excludes_identifier": RULE_CONFIG["R04"]["identifier"],
        "suggestion": "Inspect search-to-booking funnel and event records.",
    },
    "R02": {
        "conditions": "bookings >= min_bookings AND completion < completion_below",
        "excludes_identifier": RULE_CONFIG["R04"]["identifier"],
        "suggestion": "Inspect ride progression and cancellation records.",
    },
    "R03": {
        "conditions": "bookings < bookings_below",
        "suggestion": (
            "Review counts before interpreting completion rates; "
            "completion is undefined for zero bookings."
        ),
    },
    "R04": {
        "conditions": "ward_num == identifier",
        "suggestion": "Verify identifier meaning before geographic interpretation.",
    },
}

limitations = [
    "Exploratory configurable thresholds; human review required.",
    "Threshold sensitivity does not establish accuracy.",
    "All geographic mappings remain unverified.",
    "Supply timing compatibility remains unresolved.",
    "No causal, driver-shortage, uplift or operational-priority claims.",
    "Clusters and anomaly scores provide context only.",
    "No triggered rule does not establish that a record has no issues.",
]

tables = {
    "investigation_suggestions.csv": investigations,
    "record_review.csv": record_review,
    "threshold_sensitivity.csv": sensitivity,
}

for filename, table in tables.items():
    table.to_csv(EXPORT_DIR / filename, index=False)

revision = subprocess.run(
    ["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"],
    check=True, capture_output=True, text=True
).stdout.strip()

metadata = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_repository_revision": revision,
    "input_path": str(PROFILE_PATH.relative_to(PROJECT_ROOT)),
    "input_sha256": hashlib.sha256(PROFILE_PATH.read_bytes()).hexdigest(),
    "configuration": RULE_CONFIG,
    "rule_definitions": rule_definitions,
    "records_retained": len(record_review),
    "records_with_triggers": int(
        record_review["triggered_rule_count"].gt(0).sum()
    ),
    "total_triggers": len(investigations),
    "triggers_by_rule": {
        rule: int(investigations["rule_id"].eq(rule).sum())
        for rule in RULE_CONFIG
    },
    "rate_units": "fractions",
    "missing_values": "blank in CSV; not replaced with zero",
    "limitations": limitations,
    "environment": {
        "python": platform.python_version(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
    },
    "output_sha256": {
        filename: hashlib.sha256(
            (EXPORT_DIR / filename).read_bytes()
        ).hexdigest()
        for filename in tables
    },
}

(EXPORT_DIR / "metadata.json").write_text(
    json.dumps(metadata, indent=2, allow_nan=False),
    encoding="utf-8",
)

(EXPORT_DIR / "README.md").write_text(
    "# Decision & Recommendation Reports\n\n"
    "- investigation_suggestions.csv: one row per record–rule trigger.\n"
    "- record_review.csv: all input profiles plus trigger counts.\n"
    "- threshold_sensitivity.csv: one-at-a-time threshold comparisons.\n"
    "- metadata.json: rules, configuration, provenance and file hashes.\n\n"
    "Rates are fractions; missing values remain blank.\n\n"
    "## Limitations\n\n"
    + "\n".join(f"- {item}" for item in limitations)
    + "\n",
    encoding="utf-8",
)

# Verify exported CSV row counts and metadata readability.
for filename, table in tables.items():
    assert len(pd.read_csv(EXPORT_DIR / filename)) == len(table)

json.loads((EXPORT_DIR / "metadata.json").read_text())

print("Export verification passed.")
print("Saved to:", EXPORT_DIR)
for filename in [*tables, "metadata.json", "README.md"]:
    print("-", filename)

Export verification passed.
Saved to: /content/NammaFlow/reports/decision_recommendation
- investigation_suggestions.csv
- record_review.csv
- threshold_sensitivity.csv
- metadata.json
- README.md


## 8.8 — Summary

Implemented four configurable investigation rules across 244 records.
The initial configuration produced 149 triggers across 147 records:
R01: 2, R02: 76, R03: 70, and R04: 1.

All records were preserved, including three zero-booking records.
Input consistency and output integrity checks passed. Exported CSV
row counts and metadata readability were verified.

Sensitivity analysis showed material dependence on threshold choices.
Suggestions remain exploratory and require human review. Geographic
mapping and supply timing remain unresolved; operational effectiveness
has not been established.

In [10]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

zip_path = Path("/content/NammaFlow_phase8_reports.zip")

with ZipFile(zip_path, "w", ZIP_DEFLATED) as archive:
    for filename in [*tables, "metadata.json", "README.md"]:
        path = EXPORT_DIR / filename
        archive.write(path, path.relative_to(PROJECT_ROOT))

files.download(str(zip_path))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>